# Laboratorio II -- Protocolo + referencias iniciales (T0, B0)

Este notebook corre **desde el SageMaker Notebook Instance real** (Seccion 6: no se aceptan
runs desde SageMaker Studio ni Google Colab).

**Antes de correr nada, edita la celda de configuracion (la siguiente) con tus datos reales.**

Si eres quien corre el PROTOCOLO por primera vez para todo el equipo: corre todo el notebook
de arriba a abajo. Al final vas a tener un `PROTOCOL_RUN_ID` -- compartelo con tus companeros,
lo necesitan para sus propios runs.

Si ya alguien de tu equipo corrio el protocolo: salta la seccion "Paso 1" (no lo repitas -- la
muestra y los folds se fijan una sola vez para todo el equipo, Seccion 2) y usa directamente el
`PROTOCOL_RUN_ID` y el `protocol/partitions.csv` que esa persona genero.

## Configuracion -- EDITA ESTO con tus datos reales

In [5]:
# URL publica de tu Tracking Server de MLflow (la IP elastica de la EC2, puerto 5000)
MLFLOW_TRACKING_URI = "http://100.56.52.81:5000"

# Tu member_id oficial, el que asigno el curso -- usarlo SIN modificaciones (A.2)
MEMBER_ID = "E01"

# Si YA corriste (o corrio un companero) el run de protocolo, pega aqui su run_id
# y salta la seccion 'Paso 1' completa (no la vuelvas a correr).
EXISTING_PROTOCOL_RUN_ID = None  # p. ej. "a1b2c3d4..." si ya existe

## Paso 0 -- Instalar dependencias y traer el codigo del pipeline

In [6]:
%pip install -q datasets mlflow scikit-learn nltk spacy emoji
!python -m spacy download en_core_web_md -q

Note: you may need to restart the kernel to use updated packages.
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')


In [8]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))

import sentiment
print("sentiment package OK, ubicado en:", sentiment.__file__)

sentiment package OK, ubicado en: /home/ec2-user/SageMaker/Sentimientos/notebooks/../src/sentiment/__init__.py


## Paso 0.1 -- Conectar con el Tracking Server compartido y leer la procedencia

In [10]:
import mlflow
from sentiment.mlflow_contract import (
    MLFLOW_EXPERIMENT_NAME,
    read_notebook_arn,
)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

notebook_arn = read_notebook_arn()
print("Tracking URI:", mlflow.get_tracking_uri())
print("Experimento activo:", MLFLOW_EXPERIMENT_NAME)
print("notebook_arn:", notebook_arn)

Tracking URI: http://100.56.52.81:5000
Experimento activo: nlp-lab2-sentiment140
notebook_arn: arn:aws:sagemaker:us-east-1:214844251412:notebook-instance/nlp-lab2-e01


In [15]:
from sentiment.mlflow_contract import read_notebook_arn

NOTEBOOK_ARN = read_notebook_arn()
print("notebook_arn:", NOTEBOOK_ARN)

notebook_arn: arn:aws:sagemaker:us-east-1:214844251412:notebook-instance/nlp-lab2-e01


## Paso 1 -- Protocolo (SOLO si eres quien lo corre por primera vez para el equipo)

Carga el dataset real, saca la muestra estratificada de 200.000 y construye los 3 folds.
Esto puede tardar varios minutos la primera vez (descarga ~126 MB + procesamiento).

In [11]:
from sentiment.data import (
    load_train_test, build_protocol_sample_and_folds,
    save_partitions_csv, save_members_csv, resolve_columns,
)

train_ds, test_ds = load_train_test()
print("train:", len(train_ds), "test:", len(test_ds))
print("columnas:", train_ds.column_names)

TEXT_COLUMN, LABEL_COLUMN = resolve_columns(train_ds.column_names)
print("columna de texto:", TEXT_COLUMN, "| columna de label:", LABEL_COLUMN)

Generating train split: 1360000 examples [00:02, 562068.51 examples/s]
Generating test split: 240000 examples [00:00, 646285.91 examples/s]

train: 1360000 test: 240000
columnas: ['text', 'label']
columna de texto: text | columna de label: label


In [12]:
partitions_df = build_protocol_sample_and_folds(
    train_ds, text_column=TEXT_COLUMN, label_column=LABEL_COLUMN,
)
save_partitions_csv(partitions_df, "protocol/partitions.csv")
print(partitions_df["fold"].value_counts().sort_index())

fold
0    66667
1    66667
2    66666
Name: count, dtype: int64


In [16]:
TEAM_MEMBERS = [
    (MEMBER_ID, NOTEBOOK_ARN),
    # ("E02", "arn:aws:sagemaker:REGION:ACCOUNT:notebook-instance/NOTEBOOK_E02"),
    # ("E03", "arn:aws:sagemaker:REGION:ACCOUNT:notebook-instance/NOTEBOOK_E03"),
]
save_members_csv(TEAM_MEMBERS, "protocol/members.csv")

In [17]:
from sentiment.mlflow_contract import log_protocol_run

PROTOCOL_RUN_ID = log_protocol_run(
    partitions_csv_path="protocol/partitions.csv",
    members_csv_path="protocol/members.csv",
)
print("PROTOCOL_RUN_ID =", PROTOCOL_RUN_ID)
print("\nCOMPARTE este run_id con tu equipo -- lo necesitan como lab_protocol_run_id.")

🏃 View run protocol at: http://100.56.52.81:5000/#/experiments/1/runs/89341a59aa23445e877b4c91040a6ffc
🧪 View experiment at: http://100.56.52.81:5000/#/experiments/1
PROTOCOL_RUN_ID = 89341a59aa23445e877b4c91040a6ffc

COMPARTE este run_id con tu equipo -- lo necesitan como lab_protocol_run_id.


## Paso 1 (alternativo) -- Si ya existe el run de protocolo

Corre esta celda EN VEZ de la seccion de arriba si un companero ya corrio el protocolo.
Necesitas que te compartan `protocol/partitions.csv` (por ejemplo, subido al repo de GitHub,
o descargado de los artifacts del run de protocolo en la UI de MLflow).

In [ ]:
from sentiment.data import load_partitions_csv

if EXISTING_PROTOCOL_RUN_ID is not None:
    PROTOCOL_RUN_ID = EXISTING_PROTOCOL_RUN_ID
    partitions_df = load_partitions_csv("protocol/partitions.csv")  # ya deberia existir en disco
    print("Usando protocolo existente:", PROTOCOL_RUN_ID)
else:
    print("EXISTING_PROTOCOL_RUN_ID es None -- si ya corriste el Paso 1 arriba, ignora esta celda.")

## Paso 2 -- Cargar los textos/labels de la muestra completa (una sola vez)

`cv.run_cv` necesita poder indexar `texts`/`labels` por las posiciones originales de
`partitions_df['index']`. Traer las 1.360.000 filas completas de HF a listas de Python es
lento pero solo hay que hacerlo una vez por sesion de notebook.

In [18]:
import numpy as np

all_texts = train_ds[TEXT_COLUMN]
all_labels = np.asarray(train_ds[LABEL_COLUMN])
print("listo:", len(all_texts), "textos en memoria")

listo: 1360000 textos en memoria


## Paso 3 -- Referencias iniciales: T0 y B0 (Seccion 2)

Evalua T0 y B0 con el protocolo (3-fold CV) ANTES de iniciar las comparaciones por etapas.

In [19]:
from sentiment.config import t0_config, b0_config
from sentiment.cv import run_cv
from sentiment.mlflow_contract import log_experimental_run

import sklearn

t0 = t0_config(library="scikit-learn", library_version=sklearn.__version__)
t0.validate()

metrics_t0 = run_cv(t0, all_texts, all_labels, partitions_df)
print("T0:", metrics_t0)

T0: {'macro_f1_fold_0': 0.3333766636335457, 'macro_f1_fold_1': 0.3333766636335457, 'macro_f1_fold_2': 0.33337333133343333, 'macro_f1_mean': 0.33337555286684156, 'macro_f1_std': 1.5708613375925653e-06}


In [20]:
run_id_t0 = log_experimental_run(
    run_name="T0",
    config=t0,
    cv_metrics=metrics_t0,
    protocol_run_id=PROTOCOL_RUN_ID,
    experiment_id="T0",
    member_id=MEMBER_ID,
    configuration_id="CFG_T0",
    notebook_arn=NOTEBOOK_ARN,
)
print("run_id_t0 =", run_id_t0)

🏃 View run T0 at: http://100.56.52.81:5000/#/experiments/1/runs/780e664a680842a4874e070a3d3fc4b8
🧪 View experiment at: http://100.56.52.81:5000/#/experiments/1
run_id_t0 = 780e664a680842a4874e070a3d3fc4b8


In [21]:
b0 = b0_config(
    rep_library="scikit-learn", rep_version=sklearn.__version__,
    clf_library="scikit-learn", clf_version=sklearn.__version__,
)
b0.validate()

metrics_b0 = run_cv(b0, all_texts, all_labels, partitions_df)
print("B0:", metrics_b0)

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/

B0: {'macro_f1_fold_0': 0.7818645372407267, 'macro_f1_fold_1': 0.780582660372184, 'macro_f1_fold_2': 0.7818184582134228, 'macro_f1_mean': 0.7814218852754444, 'macro_f1_std': 0.0005937197138261514}


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [22]:
run_id_b0 = log_experimental_run(
    run_name="B0",
    config=b0,
    cv_metrics=metrics_b0,
    protocol_run_id=PROTOCOL_RUN_ID,
    experiment_id="B0",
    member_id=MEMBER_ID,
    configuration_id="CFG_B0",
    notebook_arn=NOTEBOOK_ARN,
)
print("run_id_b0 =", run_id_b0)

🏃 View run B0 at: http://100.56.52.81:5000/#/experiments/1/runs/dbfa1f8f09b740c08c69724b4d27f18e
🧪 View experiment at: http://100.56.52.81:5000/#/experiments/1
run_id_b0 = dbfa1f8f09b740c08c69724b4d27f18e


## Paso 4 -- Prueba del circuito completo: P_STOPWORDS

T0 y B0 NO ejercitan `nltk`/`spacy`/`emoji` (B0 tiene `stopwords=keep`, `lemmatize=false`,
`elongation=keep`, `emoji=keep`). Para validar que el circuito completo funciona de punta a
punta (SageMaker -> preprocesamiento con recursos externos -> MLflow -> artifacts) corre al
menos una comparacion real. `P_STOPWORDS` es la mas simple: 'igual a B0, salvo
stopwords=remove' (A.4).

In [23]:
from sentiment.preprocessing import load_nltk_stopwords

stopword_set = load_nltk_stopwords()  # requiere nltk instalado + internet (ya lo tienes aqui)
print(f"{len(stopword_set)} stopwords cargadas")

198 stopwords cargadas


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/ec2-user/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [24]:
p_stopwords = b0.copy()
p_stopwords.preprocessing.stopwords = "remove"
p_stopwords.validate()

metrics_p_stopwords = run_cv(
    p_stopwords, all_texts, all_labels, partitions_df, stopword_set=stopword_set,
)
print("P_STOPWORDS:", metrics_p_stopwords)

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/

P_STOPWORDS: {'macro_f1_fold_0': 0.7675013346600177, 'macro_f1_fold_1': 0.764326592534381, 'macro_f1_fold_2': 0.7644912326871073, 'macro_f1_mean': 0.765439719960502, 'macro_f1_std': 0.0014593304336617036}


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [25]:
run_id_p_stopwords = log_experimental_run(
    run_name="P_STOPWORDS",
    config=p_stopwords,
    cv_metrics=metrics_p_stopwords,
    protocol_run_id=PROTOCOL_RUN_ID,
    experiment_id="P_STOPWORDS",
    member_id=MEMBER_ID,
    configuration_id="CFG_P_STOPWORDS",
    notebook_arn=NOTEBOOK_ARN,
)
print("run_id_p_stopwords =", run_id_p_stopwords)

🏃 View run P_STOPWORDS at: http://100.56.52.81:5000/#/experiments/1/runs/fc90ebd7c0614739928526ed7b012108
🧪 View experiment at: http://100.56.52.81:5000/#/experiments/1
run_id_p_stopwords = fc90ebd7c0614739928526ed7b012108


## Listo

Revisa en la UI de MLflow (`http://TU_IP_ELASTICA:5000`) que aparezcan el run de protocolo,
`T0`, `B0` y `P_STOPWORDS` dentro del experiment `nlp-lab2-sentiment140`, cada uno con sus
tags, metricas y el artifact `run/configuration.json` (y `provenance/sagemaker-resource-
metadata.json`).

**Con esto validado:** si esta Notebook Instance era solo para probar el circuito completo
(patron 'Fase 4'), detenla o eliminala desde la consola de AWS -- no la dejes corriendo sin
uso, gasta creditos del Learner Lab.

**Siguiente paso:** el resto de las comparaciones obligatorias de la Seccion 3
(`P_STOPWORDS_NEGATION`, `P_LEMMA`, `P_ELONGATION`, `P_EMOJI`, luego
`R_BOW`/`R_TFIDF_UNI`/`R_TFIDF_UNI_BI`/`R_SPACY`, luego `C_LOGREG`/`C_LINEAR_SVM`/`C_SGD`) --
mismo patron que P_STOPWORDS: armar el config con `.copy()` y el cambio correspondiente,
correr `run_cv`, loguear con `log_experimental_run` usando el `experiment_id` exacto de cada
una.